In [1]:
import torch
import numpy as np
import matplotlib.pyplot as plt

## THE DGQ ALGORITHM

The DGQ algoithm consists of 2 components

(1) Outlier preserving group quantization for handling outliers in activations

(2) Attention aware quantization for addressing patterns in cross attention

#### Outlier-preserving group quantization

Identify outlier type by examining the activation range across channels or pixels. 
Identify the optimal dimension where outliers are most pronounced d ∈ {channel,pixel} 
Apply quantization that preserves critical outlier information

$D_d$ is a measure of variability of activation values in each dimension. 

$D_d = (\max_{\{i\}}a^{\max}_{i,d} - \min_{\{i\}}a^{\max}_{i,d}) + (\max_{\{i\}}a^{\min}_{i,d} - \min_{\{i\}}a^{\min}_{i,d})$

Where $ a^{\max}_{i,d} $ and $a^{\min}_{i,d}$ represent the maximum and minimum values of the i-th vector in dimension d, respectively. 

The largest dimension $d^*$ is where $D_d$ is the largest

$ d^* = \arg \max_{\{d\}}D_d $

At the optimal dimension $d^*$, we divide the activation balues into K groups, based on their range using K-means clustering. 

Quantization scale $s_k$ is defined $s_k = \frac{\max A - \min A}{2^b}$

Where A is the activation of the k-th group, and b is the number of quantization bits

Zero-point $z_k$ is the minimum activation of the group, $z_k = min A$.

## Implementation of DGQ with a pytorch based model

In [ ]:
#Using YOLO, lets analyse the model architecture
model = torch.hub.load('ultralytics/yolov5', 'yolov5s', pretrained=True)

Using cache found in C:\Users\anuhg/.cache\torch\hub\ultralytics_yolov5_master
YOLOv5  2026-5-7 Python-3.11.15 torch-2.11.0+cpu CPU

Fusing layers... 
YOLOv5s summary: 213 layers, 7225885 parameters, 0 gradients, 16.4 GFLOPs
Adding AutoShape... 


In [30]:
device = torch.device('cpu')

load_model = model.to(device)

for name, param in load_model.named_parameters():
    print(name, param.size())

model.model.model.0.conv.weight torch.Size([32, 3, 6, 6])
model.model.model.0.conv.bias torch.Size([32])
model.model.model.1.conv.weight torch.Size([64, 32, 3, 3])
model.model.model.1.conv.bias torch.Size([64])
model.model.model.2.cv1.conv.weight torch.Size([32, 64, 1, 1])
model.model.model.2.cv1.conv.bias torch.Size([32])
model.model.model.2.cv2.conv.weight torch.Size([32, 64, 1, 1])
model.model.model.2.cv2.conv.bias torch.Size([32])
model.model.model.2.cv3.conv.weight torch.Size([64, 64, 1, 1])
model.model.model.2.cv3.conv.bias torch.Size([64])
model.model.model.2.m.0.cv1.conv.weight torch.Size([32, 32, 1, 1])
model.model.model.2.m.0.cv1.conv.bias torch.Size([32])
model.model.model.2.m.0.cv2.conv.weight torch.Size([32, 32, 3, 3])
model.model.model.2.m.0.cv2.conv.bias torch.Size([32])
model.model.model.3.conv.weight torch.Size([128, 64, 3, 3])
model.model.model.3.conv.bias torch.Size([128])
model.model.model.4.cv1.conv.weight torch.Size([64, 128, 1, 1])
model.model.model.4.cv1.conv.bi

In [299]:
#Lets take one conv layer weights
first_layer_name, first_layer_weights = list(load_model.named_parameters())[1]

print(f"""
Layer name: {first_layer_name}\n
Layer size: {first_layer_weights.size()}""")



Layer name: model.model.model.0.conv.bias

Layer size: torch.Size([32])


For Conv layers, the weights are associated in this way: (out_channels, in_channels, kernel_size[0], kernel_size[1])

In [366]:
from sklearn.cluster import KMeans

class Quantizer:

    def __init__(self, bit_width):

        self.statistics = {}

        self.quant_bits = bit_width

    def MaxActivationDimensionCNN(self, layer):
        #Layer will be an array of weights

        #Pixel Dimension
        pixel_dim = layer.view(1,-1).squeeze()
        pixel_var = max(pixel_dim) - min(pixel_dim)
        
        #Channel Dim
        channel_dim = layer.view((layer.shape[0], -1)) #channel is dependent on the output channels
        channel_var = (max(torch.max(channel_dim, 0)[0]) - min(torch.max(channel_dim, 0)[0])) + (max(torch.min(channel_dim, 0)[0]) - min(torch.min(channel_dim, 0)[0]))
        
        if channel_var > pixel_var:
            return 1, pixel_dim
        else:
            return 0, channel_dim

    def logarithmic_quantization(self, activations, q_s, q_z):
        q_activations = torch.clamp(torch.round(-torch.log2(activations/q_s)), 0, pow(2, self.quant_bits)-1)
        return (q_activations - q_z) *q_s

    def linear_quantization(self, activations, q_s, q_z):
        q_activations = torch.clamp(torch.round(activations/q_s) + q_z , 0, pow(2, self.quant_bits)-1)
        return q_s*pow(2,-1*q_activations)        

    def Quantize(self, activations, quant_func, n_groups=2): # Can speed up later by passing in layer reference too quantise weights live
        
        q_activations = torch.zeros_like(activations)

        if len(activations.size()) < 2:
            acts = activations.unsqueeze(1)
        else:
            acts = activations

        #Group channels
        kmeans = KMeans(n_clusters=n_groups, random_state=0, n_init="auto")
        kmeans.fit(acts)
        groups = kmeans.predict(acts)
    
        quant_groups = {}

        for i in range(len(groups)):

            if groups[i] in quant_groups: #if cluster centre already mapped

                if max(acts[i])>quant_groups[groups[i]]["max"]:
                    quant_groups[groups[i]]["max"] = max(acts[i])


                elif min(acts[i])<quant_groups[groups[i]]["min"]:
                    quant_groups[groups[i]]["min"] = min(acts[i])
            
            
            else: #if cluster centre hasnt been mapped yet
                quant_groups[groups[i]] = {"min": min(acts[i]), "max": max(acts[i])}


        quant_params = {}

        for group, params in quant_groups.items():
            q_s = (params["max"] - params["min"])/(pow(2,self.quant_bits))
            z   = params["min"]
            quant_params[group] = {"q_s":q_s, "z":z}

        print(quant_params)

        #apply scaling
        for i in range(len(groups)):
            # print(activations[i])
            q_activations[i] = quant_func(activations[i], quant_params[groups[i]]["q_s"], quant_params[groups[i]]["z"])       

        return q_activations


        #For each group, calculate scale and zero point


# for name, param in load_model.named_parameters():
#     print(name, quant.MaxActivationDimensionCNN(param))

#print(quant.MaxActivationDimensionCNN(first_layer_weights))

In [383]:
num_bits   = 32
num_groups = 2

quant = Quantizer(num_bits)

print(first_layer_weights.shape)

d, dim = quant.MaxActivationDimensionCNN(first_layer_weights)
q_a = quant.Quantize(dim, quant.linear_quantization, num_groups).view(first_layer_weights.shape)

print(torch.sum(torch.pow(q_a-first_layer_weights,2)))
print(d, q_a, first_layer_weights)

torch.Size([32])
{np.int32(0): {'q_s': tensor(1.74916e-09), 'z': tensor(-3.17912)}, np.int32(1): {'q_s': tensor(0.), 'z': tensor(-10.88728)}}
tensor(282.33429)
0 tensor([1.74916e-09, 1.74916e-09, 0.00000e+00, 0.00000e+00, 0.00000e+00, 0.00000e+00, 0.00000e+00, 0.00000e+00, 0.00000e+00, 0.00000e+00, 0.00000e+00, 0.00000e+00, 0.00000e+00, 0.00000e+00, 1.74916e-09, 0.00000e+00, 1.74916e-09, 0.00000e+00, 0.00000e+00, 0.00000e+00, 0.00000e+00, 0.00000e+00, 0.00000e+00, 0.00000e+00,
        0.00000e+00, 1.74916e-09, 0.00000e+00, 0.00000e+00, 0.00000e+00, 0.00000e+00, 0.00000e+00, 0.00000e+00]) Parameter containing:
tensor([ -2.30597,  -2.34275,   1.59871,   2.40548,   1.48295,   1.40602,   1.37900,   2.34541,   2.73934,   2.29776,   1.61733,   2.61567,   2.49187,   2.35384,  -2.04709,   1.28121,  -0.79446,   2.22565,   2.29474,   1.54162,   0.84339,   1.93918,   4.33347,   3.11096,   1.47184,  -3.17912,   2.45499,   2.75382,
          2.47896, -10.88728,   2.00345,   3.32457])


In [385]:
# a =torch.tensor([[1,2,3,4],
#                  [5,6,7,8],
#                  [9,10,11,12],
#                  [13,14,15,16],
#                  [17,18,19,20]])
# b = a.view(1,-1)

num_bits   = 2
num_groups = 2

a = torch.rand((3,3,2,2))
b = torch.rand((6))

quant = Quantizer(num_bits)

d, dim = quant.MaxActivationDimensionCNN(a)
q_a = quant.Quantize(dim, quant.linear_quantization, num_groups).view(a.shape)

d, dim = quant.MaxActivationDimensionCNN(b)
q_b = quant.Quantize(dim, quant.linear_quantization, num_groups)

print(torch.sum(torch.pow(q_a-a,2)))
print(torch.sum(torch.pow(q_b-b,2)))

print(a)
print(q_a)

{np.int32(1): {'q_s': tensor(0.10328), 'z': tensor(0.01005)}, np.int32(0): {'q_s': tensor(0.11077), 'z': tensor(0.52420)}}
{np.int32(0): {'q_s': tensor(0.05082), 'z': tensor(0.09644)}, np.int32(1): {'q_s': tensor(0.03950), 'z': tensor(0.66828)}}
tensor(13.46825)
tensor(7.39648)
tensor([[[[0.31406, 0.57653],
          [0.11337, 0.58786]],

         [[0.42315, 0.90729],
          [0.96091, 0.76433]],

         [[0.86003, 0.80637],
          [0.20744, 0.41546]]],


        [[[0.82379, 0.09726],
          [0.94577, 0.34623]],

         [[0.11506, 0.81662],
          [0.61638, 0.84293]],

         [[0.58657, 0.89097],
          [0.92268, 0.61223]]],


        [[[0.06713, 0.21240],
          [0.96726, 0.41422]],

         [[0.71352, 0.56764],
          [0.05290, 0.22120]],

         [[0.82712, 0.52420],
          [0.01005, 0.53686]]]])
tensor([[[[0.01291, 0.01385],
          [0.05128, 0.01385]],

         [[0.01291, 0.01385],
          [0.01385, 0.01385]],

         [[0.01385, 0.01385],
    